In [1]:
# Pickle, Save & Load Model
import os
import pickle

def save_model(model, filename):
    with open(filename, 'wb') as file:
        pickle.dump(model, file)
        
def load_model(filename):
    if os.path.exists('filename'):
        with open(filename, 'rb') as file:
            return pickle.load(file)
    else:
        return None

In [2]:
# Read CSV
import pandas as pd

dataset = pd.read_csv('movie_sentiment.csv').sample(1000)
dataset = dataset.dropna(subset="review")
dataset = dataset.astype(str)
dataset

,title,review,sentimentScore
74670,All Is Lost,Redford delivers a tour de force performance: ...,POSITIVE
237359,Dave Chappelle's Block Party,By cutting away from the music just as things ...,NEGATIVE
125438,Seventh Son,"The word ""fantasy"" is synonymous with ""imagina...",NEGATIVE
244086,My Super Ex-Girlfriend,There are certainly worse ideas out there for ...,NEGATIVE
8069,X-Men,X-Men's myriad of positive attributes generall...,POSITIVE
...,...,...,...
209279,Things to Come,"The film is wistful, touching and funny, not l...",POSITIVE
228629,Happy Feet,Sure it all ends up amounting to little more t...,POSITIVE
162557,Starsky & Hutch,as crammed with cheese as a fondue pot on New ...,POSITIVE
242764,Baby Boy,Recommended mainly to the more devoted Singlet...,POSITIVE


In [3]:
review_list = dataset['review'].to_list()
label_list = dataset['sentimentScore'].to_list()

In [4]:
# Preprocessing
# Jadiin semua huruf kecil
# Remove punctuation
# Remove stopwords
# Stemming, lemmatization + pos_tag

import string
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import SnowballStemmer, WordNetLemmatizer
from nltk.tag import pos_tag

In [5]:
def get_tag(tag):
    if tag == 'jj':
        return 'a'
    elif tag in ['nn', 'vb', 'rb']:
        return tag[0]
    else:
        return 'n'

In [6]:
def preprocess(review_list):
    punctuation = string.punctuation
    stop_words = stopwords.words('english')
    stemmer = SnowballStemmer('english')
    lemmatizer = WordNetLemmatizer()
    
    preprocessed_list = []
    
    for review in review_list:
        review = review.lower()                                                  # Huruf kecil
        review = ''.join([char for char in review if char not in punctuation])   # Hapus punctuation
        
        tokens =  word_tokenize(review)
        tokens = [token for token in tokens if token not in stop_words]          # Hapus stopwords
        
        stemmed_tokens = [stemmer.stem(token) for token in tokens]               # Stemming
        
        tagged_tokens = pos_tag(stemmed_tokens)                                  # Lemmatization, perlu tag
        lemmatized_tokens = []
        for token, tag in tagged_tokens:
            lemmatized_token = lemmatizer.lemmatize(token, get_tag(tag.lower()))
            lemmatized_tokens.append(lemmatized_token)
        
        preprocessed_review = ' '.join(lemmatized_tokens)
        preprocessed_list.append(preprocessed_review)
    
    return preprocessed_list  

In [7]:
preprocess(review_list)

['redford deliv tour de forc perform hold screen effortless act support whatsoev',
 'cut away music thing start cook could great rockumentari becom much chappell celebr',
 'word fantasi synonym imagin thing seventh son lack',
 'certain wors idea movi comedi super exgirlfriend wayan brother hit littl man exampl howev arent mani idea poor execut one',
 'xmen myriad posit attribut general compens lessthanenthral stretch',
 'doesnt belong top rank anim fare like predecessor contain enough humor charm warmth make solid famili film',
 'taken 20 year get updat guy film prove pish vinegar made charact much fun doesnt weaken time',
 'noth el emmerich return familiar territori show evolut filmmak well intun take keep mass audienc entertain',
 'there enough wrong uneven pace haphazard storytel cartoon charact repeat run gag theyr predict longer funni etc one bound wish film better',
 'haggi manag deliv move excit film also happen verit howto guid break prison',
 'your seek movi entertain introduc

In [8]:
# Train model
from nltk.classify import NaiveBayesClassifier, accuracy
from nltk.probability import FreqDist
import random
# Semua kata di satu list
# Tentuin yang mana jadi fitur
# extract_features
# Train mode, jangan lupa disave

def train_model():
    reviews = preprocess(review_list)
    all_words = []
    
    for review in reviews:
        all_words.extend(word_tokenize(review))
    
    fd = FreqDist(all_words)
    
    word_features = [word for word, _ in fd.most_common(500)]
    
    def extract_features(review):
        tokens = word_tokenize(review)
        return {word : (word in tokens) for word in word_features}
    
    feature_set = [(extract_features(review), label) for review, label in zip(reviews, label_list)]
    
    random.shuffle(feature_set) # Agar dia random dan tidak bias
    train_length = int(len(feature_set) * 0.8)
    
    train_set = feature_set[:train_length]
    test_set = feature_set[train_length:]
    
    classifier = NaiveBayesClassifier.train(train_set)
    model_accuracy = accuracy(classifier, test_set)
    
    print("Model Accuracy: ", model_accuracy)
    save_model(classifier, "model.pickle")
    return classifier

In [9]:
train_model()

Model Accuracy:  0.6222222222222222


In [10]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

vectorizer = TfidfVectorizer(tokenizer=word_tokenize, stop_words='english', ngram_range=(1, 3))
all_words_vector = vectorizer.fit_transform(review_list)

def get_recommendation(review):
    review_vector = vectorizer.transform([review])
    similarity = cosine_similarity(review_vector, all_words_vector).flatten()
    top = similarity.argsort()[-2:]
    
    print("Top movie title: ")
    for idx in top:
        print(dataset['title'].iloc[idx])

c:\University\Semester 4\Natural Language Processing\Session 11\venv\lib\site-packages\sklearn\feature_extraction\text.py:525: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


In [11]:
get_recommendation("This movie is the best I've ever seen")

Top movie title: 
Bohemian Rhapsody
Inception


In [12]:
import spacy

def extract_ner(review):
    nlp = spacy.load('en_core_web_sm')
    # python -m spacy download en_core_web_sm <-
    
    doc = nlp(review)
    named_entities = {
        'language' : [ent.text for ent in doc.ents if ent.label_ == "LANGUAGE"],
        'location' : [ent.text for ent in doc.ents if ent.label_ in ['GPE', 'LOC']]
    }
    
    return named_entities

In [13]:
ner = extract_ner("This English movie is the best I've ever seen shot in Nile River")
print(ner['language'])
print(ner['location'])

['English']
['Nile River']
